# Feature Engineering Adaptativo para Ames Housing

En este notebook se aplican transformaciones necesarias en función del análisis EDA y los perfiles automáticos.

---

## 1. Carga de datos y separación Train/Test

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Carga
DATA_PATH = 'data/train.csv'
df = pd.read_csv(DATA_PATH)
df = df.drop('Id', axis=1)  # ya se eliminó en EDA

target = 'SalePrice'

def log_skew_warning(s):
    s = s.dropna()
    return np.abs(s.skew()) > 0.75 and (s > 0).all()

# Split train/test (90/10 por tamaño)
train, test = train_test_split(df, test_size=0.1, random_state=0)
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)
print(train.shape, test.shape)

---

## 2. Imputación de valores faltantes

### 2a. Categóricas

- Si missing > 40%: se imputa con "Missing"
- Si 5% < missing <= 40%: se imputa con "Missing"
- Si missing <= 5%: se imputa con la moda (calculada en train)

In [ ]:
# Listado de missing de EDA
a_cat_high = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType', 'FireplaceQu']
a_cat_low = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtFinType2', 'BsmtExposure', 'BsmtFinType1', 'BsmtCond', 'BsmtQual', 'Electrical']

cat_cols = ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']

for col in a_cat_high:
    for _df in [train, test]:
        _df[col] = _df[col].fillna('Missing')

for col in a_cat_low:
    moda = train[col].mode(dropna=True)[0]
    train[col] = train[col].fillna(moda)
    test[col] = test[col].fillna(moda)

---

### 2b. Numéricas

Para cada columna numérica con missing, imputar con la media (de train) y añadir indicador de missingness.

In [ ]:
# Variables numéricas con missing
a_num = ['LotFrontage', 'GarageYrBlt', 'MasVnrArea']
for col in a_num:
    # Indicador
    for _df in [train, test]:
        _df[col + '_was_missing'] = _df[col].isnull().astype(int)
    # Imputar media calculada sólo en train
    mean = train[col].mean()
    train[col] = train[col].fillna(mean)
    test[col] = test[col].fillna(mean)

---

## 3. Variables temporales (años)

Transformar a años transcurridos respecto a YrSold. Para: YearBuilt, YearRemodAdd, GarageYrBlt

In [ ]:
year_cols = ['YearBuilt','YearRemodAdd','GarageYrBlt']
for col in year_cols:
    train[f'{col}_elapsed'] = train['YrSold'] - train[col]
    test[f'{col}_elapsed'] = test['YrSold'] - test[col]

---

## 4. Variables numéricas - log/Yeo-Johnson y binarización sparce/ceros

### 4a. Log-transform o Yeo-Johnson

Transforma sólo columnas con skew alto (vista en EDA, >0.75) y positivos.

In [ ]:
from sklearn.preprocessing import PowerTransformer
skewed_cols = ['MiscVal', 'PoolArea', 'LotArea', '3SsnPorch', 'LowQualFinSF', 'KitchenAbvGr', 'BsmtFinSF2', 'ScreenPorch', 'BsmtHalfBath', 'EnclosedPorch', 'MasVnrArea', 'OpenPorchSF', 'LotFrontage', 'SalePrice', 'BsmtFinSF1', 'WoodDeckSF', 'TotalBsmtSF', 'MSSubClass', '1stFlrSF', 'GrLivArea', 'BsmtUnfSF', '2ndFlrSF']

# Quitamos SalePrice del pipeline, sólo transformar en target si ML lo pide
skewed_cols = [c for c in skewed_cols if c != 'SalePrice']

for col in skewed_cols:
    if log_skew_warning(train[col]):
        # log1p si todo es >0
        train[f'{col}_log1p'] = np.log1p(train[col])
        test[f'{col}_log1p'] = np.log1p(test[col])
    else:
        # Yeo-Johnson si hay ceros
        trans = PowerTransformer(method='yeo-johnson')
        train[f'{col}_yj'] = trans.fit_transform(train[[col]])
        test[f'{col}_yj'] = trans.transform(test[[col]])

---

### 4b. Binarización por abundancia de ceros para sparse features

In [ ]:
# >50% ceros
sparse_cols = ['MasVnrArea','BsmtFinSF2','2ndFlrSF','LowQualFinSF','BsmtFullBath','BsmtHalfBath','HalfBath','WoodDeckSF','EnclosedPorch','3SsnPorch','ScreenPorch','PoolArea','MiscVal']
for col in sparse_cols:
    for _df in [train, test]:
        _df[col + '_iszero'] = (_df[col]==0).astype(int)

---

## 5. Encoding categóricas

### 5a. Ordinales (calidad)

In [ ]:
ordinal_map = {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'Missing':0}
ordinal_vars = ['ExterQual','ExterCond','BsmtQual','BsmtCond','HeatingQC','KitchenQual','FireplaceQu','GarageQual','GarageCond','PoolQC']
for col in ordinal_vars:
    for _df in [train, test]:
        _df[col + '_num'] = _df[col].map(ordinal_map).astype(float)

---

### 5b. RareLabelEncoder en nominales de alta cardinalidad

In [ ]:
# Categóricas nominales (no ordinales)
high_card_cols = [c for c in cat_cols if train[c].nunique()>10 and c not in ordinal_vars]
def rare_label_encoder(df, cat_col, min_freq=0.01):
    freq = df[cat_col].value_counts(normalize=True)
    rare_lbl = freq[freq < min_freq].index.tolist()
    return df[cat_col].replace(rare_lbl, 'Rare')
for col in high_card_cols:
    train[col] = rare_label_encoder(train, col)
    test[col] = rare_label_encoder(train, col)   # usar train para determinar rare labels

---

### 5c. OrdinalEncoder para nominales de baja/alta cardinalidad

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
nominal_cols = [c for c in cat_cols if c not in ordinal_vars]
# Listamos columnas de string después de rare label
ord_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
ord_encoder.fit(train[nominal_cols])
train_nom_enc = pd.DataFrame(ord_encoder.transform(train[nominal_cols]), columns=[c+'_ord' for c in nominal_cols], index=train.index)
test_nom_enc = pd.DataFrame(ord_encoder.transform(test[nominal_cols]), columns=[c+'_ord' for c in nominal_cols], index=test.index)
# Unimos
test = pd.concat([test, test_nom_enc], axis=1)
train = pd.concat([train, train_nom_enc], axis=1)

---

## 6. Escalamiento

Aplica MinMaxScaler a todas las columnas numéricas exceptuando el target (SalePrice)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
# Seleccionamos columnas numéricas, excluyendo SalePrice
def get_num_cols(df):
    return [c for c in df.select_dtypes(include=[np.number]).columns if c != target]
num_cols_train = get_num_cols(train)
scaler = MinMaxScaler()
train[num_cols_train] = scaler.fit_transform(train[num_cols_train])
test[num_cols_train] = scaler.transform(test[num_cols_train])

---

## 7. Verificación de NaN/infinito

In [ ]:
assert not np.any(pd.isnull(train)), 'Train contiene NaN tras procesamiento!'
assert not np.any(pd.isnull(test)), 'Test contiene NaN tras procesamiento!'
assert np.isfinite(train.select_dtypes(include=[np.number])).all().all(), 'Valores infinitos en train!'
assert np.isfinite(test.select_dtypes(include=[np.number])).all().all(), 'Valores infinitos en test!'

print('✔️ Feature engineering completo y sin NaN/infinito en train/test.')